In [1]:
## CPU Usage Config

import os
import torch

CPU_THREADS = 4

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

torch.set_num_threads(CPU_THREADS)

print("Using CPU threads:", CPU_THREADS)

Using CPU threads: 4


In [2]:
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

# ==========================
# USER CONFIG
# ==========================
DATA_YAML = Path("/workspace/flower-classifier/config/data.yaml")
DATASET_ROOT = Path("/workspace/data/data_set/tomato-flower")
PROJECT_DIR = Path("/workspace/flower-classifier/runs")
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"   # change later if needed

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    print(f"{label}:")
    print(f"  path   = {path}")
    print(f"  exists = {path.exists()}")
    print(f"  is_dir = {path.is_dir()}")
    print(f"  is_file= {path.is_file()}")
    print()

# ==========================
# SYSTEM INFO
# ==========================
print("=== SYSTEM INFO ===")
print("Python executable:", sys.executable)
print("Python version   :", sys.version.splitlines()[0])
print("Platform         :", platform.platform())
print("Machine          :", platform.machine())
print("Processor        :", platform.processor())
print("CPU count        :", multiprocessing.cpu_count())
print("Working dir      :", Path.cwd())
print()

print("=== TORCH INFO ===")
print("torch version    :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("MPS available    :", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("torch num threads:", torch.get_num_threads())
print()

# ==========================
# ENV VARS
# ==========================
print("=== IMPORTANT ENV VARS ===")
for key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "CUDA_VISIBLE_DEVICES",
    "ULTRALYTICS_CONFIG_DIR",
]:
    print(f"{key} =", os.environ.get(key))
print()

# ==========================
# PATH CHECKS
# ==========================
print("=== PATH CHECKS ===")
print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)
    print(f"{split.upper()} COUNTS:")
    print("  image files =", count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}))
    print("  label files =", count_files(lbl_dir, {".txt"}))
    print()

# ==========================
# YAML CHECK
# ==========================
print("=== YAML CONTENTS ===")
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print(json.dumps(data_cfg, indent=2))

print()
print("=== YAML PATH VALIDATION ===")
for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    print(f"{key}: {p} | exists={p.exists()} | is_dir={p.is_dir()}")

=== SYSTEM INFO ===
Python executable: /usr/local/bin/python3.11
Python version   : 3.11.15 (main, Mar 16 2026, 23:07:56) [GCC 14.2.0]
Platform         : Linux-6.17.0-19-generic-x86_64-with-glibc2.41
Machine          : x86_64
Processor        : 
CPU count        : 8
Working dir      : /workspace/flower-classifier/notebooks

=== TORCH INFO ===
torch version    : 2.11.0+cu130
CUDA available   : False
MPS available    : False
torch num threads: 4

=== IMPORTANT ENV VARS ===
OMP_NUM_THREADS = 4
MKL_NUM_THREADS = 4
NUMEXPR_NUM_THREADS = 4
OPENBLAS_NUM_THREADS = 4
VECLIB_MAXIMUM_THREADS = None
CUDA_VISIBLE_DEVICES = None
ULTRALYTICS_CONFIG_DIR = None

=== PATH CHECKS ===
DATA_YAML:
  path   = /workspace/flower-classifier/config/data.yaml
  exists = True
  is_dir = False
  is_file= True

DATASET_ROOT:
  path   = /workspace/data/data_set/tomato-flower
  exists = True
  is_dir = True
  is_file= False

PROJECT_DIR:
  path   = /workspace/flower-classifier/runs
  exists = True
  is_dir = True
  is

In [3]:
## Verify YOLO26 is accessible and usable

from ultralytics import YOLO

try:
    YOLO("yolo26n.yaml")
    print("DONE! Model config recognized")
except Exception as e:
    print("ERROR! Model config NOT recognized")
    print(e)

DONE! Model config recognized


In [4]:
## Config loading from/saving to json helper functions

from pathlib import Path
import json

DEFAULT_LOCATION = Path("/workspace/flower-classifier/config/YOLO26_config.json")

def save_json_config(config: dict, path: str | Path = DEFAULT_LOCATION):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"DONE! Saved config to {path}")

def load_json_config(path: str | Path = DEFAULT_LOCATION) -> dict:
    path = Path(path)
    with open(path, "r") as f:
        config = json.load(f)
    print(f"DONE! Loaded config from {path}")
    return config

In [5]:
## Load parameters from specified file location

file_location = "/workspace/flower-classifier/config/YOLO26_config.json"

cfg = load_json_config(file_location)

MODEL_CFG = cfg["model_cfg"]
DATA_YAML = cfg["data_yaml"]
PROJECT_DIR = cfg["project_dir"]
RUN_NAME = cfg["run_name"]
TRAIN_PARAMS = cfg["train_params"]

print("MODEL_CFG =", MODEL_CFG)
print("DATA_YAML =", DATA_YAML)
print("PROJECT_DIR =", PROJECT_DIR)
print("RUN_NAME =", RUN_NAME)
print("TRAIN_PARAMS =", TRAIN_PARAMS)

DONE! Loaded config from /workspace/flower-classifier/config/YOLO26_config.json
MODEL_CFG = yolo26n.yaml
DATA_YAML = /workspace/flower-classifier/config/data.yaml
PROJECT_DIR = /workspace/flower-classifier/runs
RUN_NAME = tomato_flower_yolo26_from_scratch
TRAIN_PARAMS = {'epochs': 100, 'imgsz': 640, 'batch': 16, 'device': 'cpu', 'workers': 4, 'patience': 30, 'optimizer': 'auto', 'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1, 'box': 7.5, 'cls': 0.5, 'dfl': 1.5, 'close_mosaic': 10, 'single_cls': False, 'cos_lr': False, 'amp': False, 'plots': True, 'save': True, 'save_period': -1, 'exist_ok': True, 'pretrained': False, 'verbose': True}


In [6]:
## Manual overrides

## for example:
# cfg["run_name"] = "tomato_flower_yolo26_trial_02"
# cfg["train_params"]["epochs"] = 150

## We change the configs within the 'cfg' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file if we wish.

In [ ]:
## Training the YOLO26 model

from ultralytics import YOLO
model = YOLO(cfg["model_cfg"])

results = model.train(
    data=cfg["data_yaml"],
    project=cfg["project_dir"],
    name=cfg["run_name"],
    **cfg["train_params"]
)

print("DONE! Training complete!")

Ultralytics 8.4.30 🚀 Python-3.11.15 torch-2.11.0+cu130 CPU (AMD Ryzen 5 3400G with Radeon Vega Graphics)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/flower-classifier/config/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tomato_flower_yolo26_from_scratch, nbs=64, nms=False, opset=None, optimize

In [ ]:
## Save current config values backk to specified config json file.

CONFIG_PATH = "/workspace/flower-classifier/config/YOLO26_config.json"
save_json_config(cfg, CONFIG_PATH)